# Lesson 2.3 - Probability & Statistics (Titanic dataset)

## Objectives
- Estimate key probabilities from real tabular data.
- Use descriptive statistics and conditional probabilities.
- Connect uncertainty analysis to practical risk scenarios.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

TITANIC_URL = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv'

try:
    titanic_df = pd.read_csv(TITANIC_URL)
    print('Loaded shape:', titanic_df.shape)
except Exception as e:
    print('Failed to load Titanic data:', e)
    raise

## Section A - Basic Stats

In [ ]:
print(titanic_df[['survived', 'age', 'fare']].describe())

print('
Survival probability (marginal):')
print(titanic_df['survived'].value_counts(normalize=True))

## Section B - Conditional Probability

In [ ]:
def conditional_survival(df, condition_col, condition_val):
    sub = df[df[condition_col] == condition_val]
    return sub['survived'].mean()

print('P(survived | sex=female):', conditional_survival(titanic_df, 'sex', 'female'))
print('P(survived | sex=male):', conditional_survival(titanic_df, 'sex', 'male'))

print('
P(survived | class):')
print(titanic_df.groupby('class')['survived'].mean())

## Section C - Distribution Exploration

In [ ]:
print('Age percentiles:', titanic_df['age'].quantile([0.25, 0.5, 0.75]).to_dict())
print('Fare percentiles:', titanic_df['fare'].quantile([0.25, 0.5, 0.75]).to_dict())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(titanic_df['age'].dropna(), bins=20, ax=axes[0])
axes[0].set_title('Age Distribution')
sns.boxplot(x=titanic_df['fare'], ax=axes[1])
axes[1].set_title('Fare Distribution')
plt.tight_layout()
plt.show()

## Section D - Simple Risk View

In [ ]:
risk_df = titanic_df.copy()
risk_df['risk_score'] = (
    (risk_df['sex'] == 'male').astype(int) * 2
    + risk_df['class'].map({'First': 0, 'Second': 1, 'Third': 2}).fillna(1).astype(int)
)

print(risk_df[['risk_score', 'survived']].groupby('risk_score')['survived'].mean())

## Section E - Data Issues & Checks

In [ ]:
required_cols = {'survived', 'sex', 'class', 'age', 'fare'}
missing_cols = required_cols.difference(titanic_df.columns)
if missing_cols:
    raise ValueError(f'Missing required columns: {missing_cols}')

print('Missing values per core column:')
print(titanic_df[['survived', 'sex', 'class', 'age', 'fare']].isna().sum())

## Business Logic & Exceptions

Probability and statistics help translate raw events into risk estimates and decision policies. In production systems, data drift, missing values, or sampling bias can distort probability estimates and downstream business decisions.

## Interview Questions & Answers

1. **Q: What is conditional probability?**  
   **A:** Probability of an event given that another event has occurred.

2. **Q: Why use normalized value counts?**  
   **A:** They convert counts into probabilities for easier interpretation.

3. **Q: What is class imbalance?**  
   **A:** Uneven label distribution that can bias naive analysis and models.

4. **Q: Correlation vs causation?**  
   **A:** Correlation indicates association, not direct causal effect.

5. **Q: Why inspect percentiles, not only mean?**  
   **A:** Percentiles capture distribution spread and outlier behavior.

6. **Q: Why check missing values early?**  
   **A:** Missingness can bias estimates and break downstream processing.

7. **Q: How can probability help business decisions?**  
   **A:** It quantifies uncertainty for prioritization and resource planning.

8. **Q: What does P(survived | class) tell us?**  
   **A:** Survival likelihood conditioned on passenger class segment.

9. **Q: Why might group probabilities change in production?**  
   **A:** Population drift and data collection changes alter underlying distributions.

10. **Q: Why is uncertainty important in AI systems?**  
    **A:** Decisions under uncertainty require calibrated risk interpretation.